# Tema de Programare: Construirea Support Vector Machines de la Zero

Bun venit la tema despre clasificarea cu Support Vector Machine (SVM).

Support Vector Machines se numara printre cei mai puternici si mai versatili algoritmi de invatare supervizata. Desi retelele neuronale au dominat titlurile in ultimii ani, SVM-urile raman alegerea de baza pentru multe aplicatii practice datorita fundamentului lor teoretic solid, performantelor excelente pe seturi de date de dimensiune medie si capacitatii de a functiona eficient in spatii cu dimensionalitate mare.

In aceasta tema, vei implementa clasificatori SVM folosind atat scikit-learn, cat si implementari de la zero cu NumPy. Vei lucra cu frontiere de decizie liniare si neliniare, vei explora impactul hiperparametrilor si vei intelege de ce SVM-urile sunt deosebit de eficiente pentru sarcini complexe de clasificare.

SVM-urile functioneaza prin gasirea hiperplanului optim care maximizeaza marginea dintre clase. "Vectorii-suport" sunt punctele de date critice aflate cel mai aproape de frontiera de decizie, care definesc aceasta margine. Aceasta abordare geometrica face ca SVM-urile sa fie in mod natural robuste si, de multe ori, sa necesite mai putine exemple de antrenare decat alti algoritmi.

**Ce vei face in aceasta tema**

* Vei intelege fundamentul matematic al SVM-urilor, inclusiv maximizarea marginii si trucul kernelului.
* Vei implementa Linear SVM pentru date liniar separabile si vei vizualiza vectorii-suport si marginile.
* Vei demonstra importanta critica a scalarii trasaturilor pentru performanta SVM.
* Vei aplica metode bazate pe kernel (RBF) pentru a trata date neliniar separabile.
* Vei compara kerneluri diferite (liniar, polinomial, RBF) si vei intelege cand exceleaza fiecare.
* Vei implementa strategii de clasificare multi-clasa (One-vs-Rest si One-vs-One).
* Vei construi un Linear SVM complet de la zero folosind gradient descent.
* Vei ajusta hiperparametrii (C, gamma) si vei intelege impactul lor asupra comportamentului modelului.

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* Toate celulele sunt inghetate, cu exceptia celor in care trebuie sa trimiti solutiile tale sau atunci cand este mentionat explicit ca poti interactiona cu ele.

* In fiecare celula de exercitiu, cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde trebuie sa scrii codul-solutie. **Nu adauga si nu modifica niciun cod din afara acestor comentarii**.

* Poti adauga celule noi pentru experimente, dar acestea vor fi omise de grader, asa ca nu te baza pe celulele nou create pentru a gazdui codul-solutie; foloseste locurile deja oferite pentru asta.

* Evita sa folosesti variabile globale daca nu este absolut necesar. Grader-ul iti testeaza codul intr-un mediu izolat, fara sa ruleze toate celulele de sus in jos. Ca rezultat, variabilele globale pot fi indisponibile in momentul evaluarii. Variabilele globale care trebuie folosite vor fi definite cu MAJUSCULE.

* Pentru a trimite notebook-ul la evaluare, salveaza-l mai intai facand click pe iconita 💾 din coltul stanga-sus al paginii, apoi apasa pe butonul `Submit assignment` din coltul dreapta-sus al paginii.
---


## Cuprins
- [Importuri](#imports)
- [1 - Introducere in Support Vector Machines](#1)
    - [1.1 - Principiul marginii maxime](#1-1)
    - [1.2 - Vectorii-suport si marginea](#1-2)
    - [1.3 - Trucul kernelului](#1-3)
- [2 - SVM liniar pentru date separabile](#2)
    - [2.1 - Pregatirea setului de date](#2-1)
    - **[Exercitiul 1 - train_linear_svm](#ex-1)**
    - [2.2 - Vizualizarea frontierelor de decizie si a vectorilor-suport](#2-2)
- [3 - Impactul scalarii trasaturilor](#3)
    - **[Exercitiul 2 - compare_scaled_unscaled_svm](#ex-2)**
- [4 - SVM cu kernel RBF](#4)
    - [4.1 - Intelegerea kernelului RBF](#4-1)
    - **[Exercitiul 3 - tune_rbf_svm](#ex-3)**
- [5 - Selectia si compararea kernelurilor](#5)
    - **[Exercitiul 4 - compare_kernels](#ex-4)**
- [6 - Clasificare multi-clasa](#6)
    - [6.1 - One-vs-Rest vs One-vs-One](#6-1)
    - **[Exercitiul 5 - train_multiclass_svm_strategies](#ex-5)**
- [7 - SVM liniar de la zero](#7)
    - [7.1 - Formularea primala](#7-1)
    - **[Exercitiul 6a - hinge_loss](#ex-6a)**
    - **[Exercitiul 6b - svm_primal_gradients](#ex-6b)**
    - **[Exercitiul 6c - train_linear_svm_from_scratch](#ex-6c)**


<a name='imports'></a>
## Importuri


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests

from sklearn.datasets import make_blobs, make_moons, make_circles, load_digits
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

<a name='1'></a>
## 1 - Introducere in Support Vector Machines

Support Vector Machines (SVM) sunt algoritmi puternici de invatare supervizata care exceleaza in sarcini de clasificare. Spre deosebire de algoritmii care incearca doar sa separe clasele, SVM-urile urmaresc sa gaseasca **hiperplanul de separare optim** care maximizeaza distanta (marginea) dintre clase.


<a name='1-1'></a>
### 1.1 - Principiul marginii maxime

Ideea-cheie din spatele SVM-urilor este **maximizarea marginii**. Dat un set de date liniar separabil, exista infinit de multe hiperplane care pot separa clasele. SVM alege hiperplanul care maximizeaza distanta minima pana la orice punct de antrenare.

Pentru un clasificator liniar, functia de decizie este:

$$f(x) = w^T x + b$$

unde $w$ este vectorul de ponderi (perpendicular pe hiperplan) iar $b$ este termenul de bias.

**Marginea** este definita astfel:

$$\text{margin} = \frac{2}{\|w\|}$$

Pentru a maximiza marginea, minimizam $\|w\|^2$, sub constrangerea ca toate punctele sa fie clasificate corect:

$$y_i(w^T x_i + b) \geq 1, \quad \forall i$$

Acest lucru conduce la **problema de optimizare hard-margin SVM**:

$$\min_{w,b} \frac{1}{2}\|w\|^2$$
$$\text{subject to: } y_i(w^T x_i + b) \geq 1, \quad \forall i$$


<a name='1-2'></a>
### 1.2 - Vectorii-suport si marginea

**Vectorii-suport** sunt exemplele de antrenare care se afla exact pe limitele marginii. Acestea sunt punctele critice care definesc frontiera de decizie. In mod remarcabil, hiperplanul optim depinde doar de acesti vectori-suport; alte puncte de antrenare ar putea fi eliminate fara sa schimbe solutia!

Pentru date din lumea reala care nu sunt perfect separabile, folosim **soft-margin SVM**, care permite unele clasificari gresite prin introducerea variabilelor de slack $\xi_i$:

$$\min_{w,b,\xi} \frac{1}{2}\|w\|^2 + C\sum_{i=1}^{m}\xi_i$$
$$\text{subject to: } y_i(w^T x_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0$$

Parametrul $C$ controleaza compromisul dintre:
- **C mare**: Margine mai mica, mai putine clasificari gresite (risc de overfitting)
- **C mic**: Margine mai mare, mai multe clasificari gresite (risc de underfitting)


<a name='1-3'></a>
### 1.3 - Trucul kernelului

Pentru date neliniar separabile, putem transforma trasaturile intr-un spatiu cu dimensionalitate mai mare, unde devin separabile. **Trucul kernelului** ne permite sa calculam produsele scalare in acest spatiu de dimensionalitate mare fara sa calculam explicit transformarea!

Kernelurile uzuale includ:

**Kernel liniar**: $K(x_i, x_j) = x_i^T x_j$

**Kernel polinomial**: $K(x_i, x_j) = (x_i^T x_j + c)^d$

**Kernel RBF (Radial Basis Function)**: $K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$

Kernelul RBF este deosebit de puternic; poate aproxima orice functie continua si functioneaza bine atunci cand nu exista cunostinte anterioare despre distributia datelor.


<a name='helpers'></a>
## Functii ajutatoare

Vom defini o functie utilitara pentru vizualizarea frontierelor de decizie in 2D.


In [ ]:
def plot_decision_boundary_2d(model, X, y, title="Decision Boundary", h=0.02):
    """Plot a 2D decision boundary."""
    X = np.asarray(X)
    y = np.asarray(y)

    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(9, 7))
    plt.contourf(xx, yy, Z, alpha=0.25)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors="k", s=35)
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()

<a name='2'></a>
## 2 - SVM liniar pentru date separabile

Sa incepem prin a implementa un SVM liniar pe un set de date aproximativ liniar separabil. Acest lucru ne va ajuta sa intelegem conceptele de baza inainte de a trece la scenarii mai complexe.


<a name='2-1'></a>
### 2.1 - Pregatirea setului de date

Vom crea un set de date sintetic cu doua clustere bine separate folosind functia `make_blobs` din scikit-learn.


In [ ]:
# Create a linearly-separable dataset
X_lin, y_lin = make_blobs(n_samples=400, centers=2, cluster_std=1.2, random_state=42)
y_lin = y_lin.astype(int)

plt.figure(figsize=(7,6))
plt.scatter(X_lin[:,0], X_lin[:,1], c=y_lin, edgecolors="k", s=30)
plt.title("Blobs Dataset (Approximately Linearly Separable)")
plt.xlabel("x1"); plt.ylabel("x2")
plt.show()

X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(
    X_lin, y_lin, test_size=0.25, random_state=42, stratify=y_lin
)

print(f"Training samples: {X_train_lin.shape[0]}")
print(f"Testing samples: {X_test_lin.shape[0]}")
print(f"Number of features: {X_train_lin.shape[1]}")

<a name='ex-1'></a>
#### **Exercitiul 1 - `train_linear_svm`**

**Sarcina ta:**

Implementeaza functia `train_linear_svm` care antreneaza un clasificator Support Vector Machine liniar folosind clasa `SVC` din scikit-learn.

Trebuie sa implementezi:

* **Creeaza o instanta SVC cu kernel liniar**:
    * Foloseste `SVC(kernel='linear', C=C)` pentru a crea modelul.
    * Parametrul `C` controleaza forta regularizarii (inversa regularizarii).

* **Antreneaza modelul pe datele de train**:
    * Foloseste metoda `.fit()` pentru a antrena pe trasaturile si etichetele oferite.

* **Returneaza modelul antrenat**:
    * Returneaza obiectul SVC deja antrenat pentru predictii si analiza ulterioara.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru crearea modelului:**
* Nu este nevoie de import suplimentar (este deja importat sus): `from sklearn.svm import SVC`.
* Creeaza modelul: `model = SVC(kernel='linear', C=C)`.

**Pentru antrenare:**
* Antreneaza modelul: `model.fit(X, y)`.
* Metoda `fit` modifica modelul in-place si il returneaza de asemenea.

**Pentru returnare:**
* Returneaza pur si simplu obiectul model deja antrenat.

</details>


In [ ]:
# GRADED FUNCTION: train_linear_svm
def train_linear_svm(X, y, C=1.0):
    """
    Train a linear SVM classifier.

    Args:
        X: Feature array of shape (m, n_features)
        y: Target array of shape (m,)
        C: Regularization parameter (default: 1.0)

    Returns:
        model: Trained SVC with linear kernel
    """
    ### ÎNCEPUT DE COD AICI ###
    
    # Create SVC with linear kernel
    
    # Fit the model on training data
    
    ### SFÂRȘIT DE COD AICI ###
    
    return model

In [ ]:
# Test your implementation
model_lin = train_linear_svm(X_train_lin, y_train_lin, C=1.0)
y_pred = model_lin.predict(X_test_lin)

print("Test Accuracy:", accuracy_score(y_test_lin, y_pred))
print("\nClassification Report:")
print(classification_report(y_test_lin, y_pred))

print(f"\nNumber of support vectors: {len(model_lin.support_vectors_)}")
print(f"Support vectors per class: {model_lin.n_support_}")

##### **Rezultatul asteptat**
```
Test Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        50
           1       1.00      1.00      1.00        50

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100

Number of support vectors: 6
Support vectors per class: [3 3]
```


In [ ]:
unittests.exercise_1(train_linear_svm)

<a name='2-2'></a>
### 2.2 - Vizualizarea frontierelor de decizie si a vectorilor-suport

Unul dintre cele mai atractive aspecte ale SVM-urilor este interpretabilitatea lor. Putem vizualiza frontiera de decizie si putem identifica ce puncte de antrenare sunt vectori-suport, adica punctele critice care definesc marginea.


In [ ]:
# Visualize decision boundary
plot_decision_boundary_2d(model_lin, X_train_lin, y_train_lin, title="Linear SVM Decision Boundary")

# Highlight support vectors
sv = model_lin.support_vectors_
plt.figure(figsize=(9,7))
plt.scatter(X_train_lin[:,0], X_train_lin[:,1], c=y_train_lin, edgecolors="k", s=35, alpha=0.6)
plt.scatter(sv[:,0], sv[:,1], s=180, facecolors="none", edgecolors="red", linewidths=2.5, label="Support Vectors")
plt.title("Support Vectors Highlighted")
plt.xlabel("x1"); plt.ylabel("x2")
plt.legend()
plt.show()

print(f"Only {len(sv)} out of {len(X_train_lin)} training points are support vectors!")
print("These critical points alone define the entire decision boundary.")

<a name='3'></a>
## 3 - Impactul scalarii trasaturilor

SVM-urile sunt foarte sensibile la scala trasaturilor deoarece se bazeaza pe calcule de distanta. Trasaturile cu scale mai mari vor domina calculul distantei, ceea ce duce la frontiere de decizie suboptime. Hai sa demonstram aceasta problema critica.


<a name='ex-2'></a>
#### **Exercitiul 2 - `compare_scaled_unscaled_svm`**

**Sarcina ta:**

Implementeaza functia `compare_scaled_unscaled_svm` care antreneaza un SVM cu kernel RBF atat pe trasaturi nescalate, cat si pe trasaturi scalate, pentru a demonstra importanta scalarii trasaturilor.

Trebuie sa implementezi:

* **Antreneaza pe date nescalate**:
    * Creeaza si antreneaza un SVC cu kernel RBF pe datele brute de antrenare.
    * Prezice pe datele de test nescalate si calculeaza acuratetea.

* **Scaleaza datele**:
    * Foloseste `StandardScaler()` pentru a normaliza trasaturile (medie zero, varianta unitara).
    * Antreneaza scaler-ul doar pe datele de train, apoi transforma atat train, cat si test.

* **Antreneaza pe date scalate**:
    * Creeaza si antreneaza un SVC cu kernel RBF pe datele de train scalate.
    * Prezice pe datele de test scalate si calculeaza acuratetea.

* **Returneaza rezultatele**:
    * Returneaza un dictionar cu ambele acurateti, ambele modele si scaler-ul.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru modelul nescalat:**
* Creeaza: `unscaled_model = SVC(kernel='rbf', C=C, gamma=gamma)`.
* Antreneaza: `unscaled_model.fit(X_train, y_train)`.
* Prezice si evalueaza: `unscaled_acc = accuracy_score(y_test, unscaled_model.predict(X_test))`.

**Pentru scalare:**
* Creeaza scaler-ul: `scaler = StandardScaler()`.
* Fit si transform pe train: `X_train_scaled = scaler.fit_transform(X_train)`.
* Transform doar pe test: `X_test_scaled = scaler.transform(X_test)`.

**Pentru modelul scalat:**
* La fel ca modelul nescalat, dar foloseste datele scalate.

**Pentru dictionarul returnat:**
* `results = {'unscaled_acc': ..., 'scaled_acc': ..., 'unscaled_model': ..., 'scaled_model': ..., 'scaler': ...}`.

</details>


In [ ]:
# Create a dataset with feature scale imbalance
X_scale, y_scale = make_moons(n_samples=400, noise=0.25, random_state=42)
y_scale = y_scale.astype(int)

# Artificially create scale imbalance (multiply one feature by 25)
X_imbalanced = X_scale.copy()
X_imbalanced[:, 1] *= 25.0

X_tr, X_te, y_tr, y_te = train_test_split(
    X_imbalanced, y_scale, test_size=0.25, random_state=42, stratify=y_scale
)

print(f"Feature 1 range: [{X_tr[:, 0].min():.2f}, {X_tr[:, 0].max():.2f}]")
print(f"Feature 2 range: [{X_tr[:, 1].min():.2f}, {X_tr[:, 1].max():.2f}]")
print("\nNotice the huge scale difference!")

In [ ]:
# GRADED FUNCTION: compare_scaled_unscaled_svm
def compare_scaled_unscaled_svm(X_train, X_test, y_train, y_test, C=1.0, gamma='scale'):
    """
    Train RBF SVM on unscaled vs scaled features and compare performance.

    Args:
        X_train: Training features
        X_test: Test features
        y_train: Training labels
        y_test: Test labels
        C: Regularization parameter
        gamma: Kernel coefficient

    Returns:
        results: dict with keys:
            'unscaled_acc': float - Accuracy on unscaled data
            'scaled_acc': float - Accuracy on scaled data
            'unscaled_model': trained SVC on unscaled data
            'scaled_model': trained SVC on scaled data
            'scaler': fitted StandardScaler
    """
    results = {}
    
    ### ÎNCEPUT DE COD AICI ###
    
    # Train on unscaled data
    
    # Scale the data
    
    # Train on scaled data
    
    ### SFÂRȘIT DE COD AICI ###
    
    return results

In [ ]:
# Test your implementation
results_scale = compare_scaled_unscaled_svm(X_tr, X_te, y_tr, y_te, C=1.0, gamma='scale')

print("="*60)
print("FEATURE SCALING IMPACT")
print("="*60)
print(f"Unscaled accuracy: {results_scale['unscaled_acc']:.4f}")
print(f"Scaled accuracy:   {results_scale['scaled_acc']:.4f}")
print(f"\nImprovement: {(results_scale['scaled_acc'] - results_scale['unscaled_acc'])*100:.1f}%")
print("="*60)

##### **Rezultatul asteptat**
```
============================================================
FEATURE SCALING IMPACT
============================================================
Unscaled accuracy: 0.5000
Scaled accuracy:   0.9200

Improvement: 42.0%
============================================================
```


In [ ]:
unittests.exercise_2(compare_scaled_unscaled_svm)

In [ ]:
# Visualize the difference
plot_decision_boundary_2d(results_scale["unscaled_model"], X_tr, y_tr, title="RBF SVM (Unscaled Data)")

X_tr_scaled = results_scale["scaler"].transform(X_tr)
plot_decision_boundary_2d(results_scale["scaled_model"], X_tr_scaled, y_tr, title="RBF SVM (Scaled Data)")

<a name='4'></a>
## 4 - SVM cu kernel RBF

Pentru date care nu sunt liniar separabile, avem nevoie de frontiere de decizie neliniare. Kernelul RBF (Radial Basis Function) este una dintre cele mai populare alegeri pentru aceasta sarcina.


<a name='4-1'></a>
### 4.1 - Intelegerea kernelului RBF

Kernelul RBF este definit astfel:

$$K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$$

Parametrul $\gamma$ controleaza influenta fiecarui exemplu de antrenare:
- **$\gamma$ mare**: Fiecare punct de antrenare are o raza mica de influenta → frontiere mai complexe (risc de overfitting)
- **$\gamma$ mic**: Fiecare punct de antrenare are o raza mare de influenta → frontiere mai netede (risc de underfitting)

Impreuna cu parametrul de regularizare $C$, avem doi hiperparametri de ajustat pentru performanta optima.


In [ ]:
# Create a non-linearly separable dataset (moons)
X_moons, y_moons = make_moons(n_samples=400, noise=0.2, random_state=42)
y_moons = y_moons.astype(int)

plt.figure(figsize=(7,6))
plt.scatter(X_moons[:,0], X_moons[:,1], c=y_moons, edgecolors="k", s=30)
plt.title("Moons Dataset (Non-linearly Separable)")
plt.xlabel("x1"); plt.ylabel("x2")
plt.show()

print("This dataset cannot be separated by a straight line!")

<a name='ex-3'></a>
#### **Exercitiul 3 - `tune_rbf_svm`**

**Sarcina ta:**

Implementeaza functia `tune_rbf_svm` care foloseste `GridSearchCV` pentru a gasi cei mai buni hiperparametri (C si gamma) pentru un SVM cu kernel RBF.

Trebuie sa implementezi:

* **Configureaza grila de parametri**:
    * Daca nu sunt furnizate, foloseste intervale implicite rezonabile pentru C si gamma.
    * Exemplu: `C_values = [0.1, 1, 10, 100]` si `gamma_values = [0.001, 0.01, 0.1, 1]`.

* **Creeaza `GridSearchCV`**:
    * Foloseste `GridSearchCV` cu un `SVC(kernel='rbf')`.
    * Foloseste validare incrucisata in 5 fold-uri pentru a evalua fiecare combinatie de parametri.

* **Antreneaza si extrage rezultatele**:
    * Antreneaza grid search-ul pe datele de train.
    * Extrage cel mai bun model, cei mai buni parametri si cel mai bun scor de cross-validation.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru grila de parametri:**
* Verifica daca sunt furnizate: `if C_values is None: C_values = [0.1, 1, 10, 100]`.
* Creeaza dictionarul: `param_grid = {'C': C_values, 'gamma': gamma_values}`.

**Pentru `GridSearchCV`:**
* Creeaza modelul de baza: `svc = SVC(kernel='rbf')`.
* Creeaza cautarea in grila: `grid = GridSearchCV(svc, param_grid, cv=5, scoring='accuracy')`.
* Antreneaza: `grid.fit(X, y)`.

**Pentru extragerea rezultatelor:**
* Cel mai bun model: `best_model = grid.best_estimator_`.
* Cei mai buni parametri: `best_params = grid.best_params_`.
* Cel mai bun scor: `best_score = grid.best_score_`.

</details>


In [ ]:
# GRADED FUNCTION: tune_rbf_svm
def tune_rbf_svm(X, y, C_values=None, gamma_values=None):
    """
    Tune RBF SVM hyperparameters using GridSearchCV.

    Args:
        X: Feature array
        y: Target array
        C_values: list of C values to try (optional)
        gamma_values: list of gamma values to try (optional)

    Returns:
        best_model: trained SVC with RBF kernel using best parameters
        best_params: dict of best parameters found
        best_score: float - best cross-validation score
    """
    ### ÎNCEPUT DE COD AICI ###
    
    # Set default parameter values if not provided
    
    # Create parameter grid
    
    # Create and fit GridSearchCV
    
    # Extract best results
    
    ### SFÂRȘIT DE COD AICI ###
    
    return best_model, best_params, best_score

In [ ]:
# Test your implementation
best_rbf, best_params, best_cv = tune_rbf_svm(X_moons, y_moons)

print("="*60)
print("RBF SVM HYPERPARAMETER TUNING")
print("="*60)
print(f"Best parameters: {best_params}")
print(f"Best CV score: {best_cv:.4f}")
print("="*60)

# Visualize the result
plot_decision_boundary_2d(best_rbf, X_moons, y_moons, title="Best RBF SVM (Tuned Hyperparameters)")

# Highlight support vectors
sv = best_rbf.support_vectors_
plt.figure(figsize=(9,7))
plt.scatter(X_moons[:,0], X_moons[:,1], c=y_moons, edgecolors="k", s=35, alpha=0.6)
plt.scatter(sv[:,0], sv[:,1], s=180, facecolors="none", edgecolors="red", linewidths=2.5, label="Support Vectors")
plt.title("RBF SVM Support Vectors (Non-linear Boundary)")
plt.xlabel("x1"); plt.ylabel("x2")
plt.legend()
plt.show()

##### **Rezultatul asteptat**
```
============================================================
RBF SVM HYPERPARAMETER TUNING
============================================================
Best parameters: {'C': X, 'gamma': X.XXX}
Best CV score: 0.9XXX
============================================================
```


In [ ]:
unittests.exercise_3(tune_rbf_svm)

<a name='5'></a>
## 5 - Selectia si compararea kernelurilor

Seturi de date diferite au caracteristici diferite, iar niciun kernel nu functioneaza cel mai bine pentru toate problemele. Hai sa comparam trei kerneluri populare pe mai multe seturi de date pentru a intelege punctele forte si slabiciunile fiecaruia.


<a name='ex-4'></a>
#### **Exercitiul 4 - `compare_kernels`**

**Sarcina ta:**

Implementeaza functia `compare_kernels` care compara kernelurile liniar, polinomial si RBF pe mai multe seturi de date.

Trebuie sa implementezi:

* **Parcurge seturile de date**:
    * Pentru fiecare set de date din dictionar, extrage trasaturile si etichetele.
    * Imparte in train/test pentru o evaluare corecta.

* **Antreneaza cele trei tipuri de kernel**:
    * Kernel liniar: `SVC(kernel='linear', C=C)`.
    * Kernel polinomial: `SVC(kernel='poly', C=C, degree=degree, gamma=gamma)`.
    * Kernel RBF: `SVC(kernel='rbf', C=C, gamma=gamma)`.

* **Evalueaza si stocheaza rezultatele**:
    * Calculeaza acuratetea pe test pentru fiecare kernel pe fiecare set de date.
    * Stocheaza rezultatele intr-un DataFrame pandas, cu seturile de date pe linii si kernelurile pe coloane.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru parcurgerea seturilor de date:**
* Itereaza: `for name, (X, y) in datasets.items()`.
* Imparte: `X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)`.

**Pentru fiecare kernel:**
* Creeaza, antreneaza, prezice:
  ```python
  model = SVC(kernel='linear', C=C)
  model.fit(X_train, y_train)
  acc = accuracy_score(y_test, model.predict(X_test))
  ```

**Pentru stocarea rezultatelor:**
* Foloseste un dictionar: `results = {dataset_name: {kernel_name: accuracy}}`.
* Converteste in DataFrame: `results_df = pd.DataFrame(results).T`.

</details>


In [ ]:
# Create three different datasets
X_circles, y_circles = make_circles(n_samples=400, noise=0.12, factor=0.5, random_state=42)
y_circles = y_circles.astype(int)

datasets = {
    "blobs": (X_lin, y_lin),
    "moons": (X_moons, y_moons),
    "circles": (X_circles, y_circles),
}

# Visualize all datasets
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (name, (X, y)) in enumerate(datasets.items()):
    axes[idx].scatter(X[:,0], X[:,1], c=y, edgecolors="k", s=30)
    axes[idx].set_title(f"{name.capitalize()} Dataset")
    axes[idx].set_xlabel("x1")
    axes[idx].set_ylabel("x2")
plt.tight_layout()
plt.show()

In [ ]:
# GRADED FUNCTION: compare_kernels
def compare_kernels(datasets, C=1.0, degree=3, gamma='scale'):
    """
    Compare linear, polynomial, and RBF kernels across multiple datasets.

    Args:
        datasets: dict mapping dataset name -> (X, y) tuple
        C: Regularization parameter
        degree: Degree for polynomial kernel
        gamma: Kernel coefficient for poly and RBF

    Returns:
        results_df: pandas DataFrame with datasets as rows and kernels as columns
                   Each cell contains the test accuracy
    """
    ### ÎNCEPUT DE COD AICI ###
    
    # Initialize results dictionary
    
    # Loop through each dataset
    
        # Split into train/test
        
        # Train and evaluate linear kernel
        
        # Train and evaluate polynomial kernel
        
        # Train and evaluate RBF kernel
        
    # Convert results to DataFrame
    
    ### SFÂRȘIT DE COD AICI ###
    
    return results_df

In [ ]:
# Test your implementation
results_df = compare_kernels(datasets, C=1.0, degree=3, gamma='scale')

print("="*60)
print("KERNEL COMPARISON ACROSS DATASETS")
print("="*60)
print(results_df)
print("="*60)
print("\nKey Insights:")
print("• Linear kernel excels on linearly separable data (blobs)")
print("• RBF kernel is versatile and performs well on all datasets")
print("• Polynomial kernel can struggle without proper tuning")

##### **Rezultatul asteptat**
```
============================================================
KERNEL COMPARISON ACROSS DATASETS
============================================================
          linear  polynomial       rbf
blobs     1.0000      0.XXXX    1.0000
moons     0.8XXX      0.XXXX    0.9XXX
circles   0.5XXX      0.XXXX    0.9XXX
============================================================
```


In [ ]:
unittests.exercise_4(compare_kernels)

<a name='6'></a>
## 6 - Clasificare multi-clasa

SVM-urile sunt in mod inerent clasificatori binari. Pentru probleme multi-clasa, avem nevoie de strategii care combina mai multi clasificatori binari. Cele mai comune doua abordari sunt One-vs-Rest (OvR) si One-vs-One (OvO).


<a name='6-1'></a>
### 6.1 - One-vs-Rest vs One-vs-One

**One-vs-Rest (OvR)**:
- Antreneaza $K$ clasificatori binari (cate unul pentru fiecare clasa)
- Fiecare clasificator separa o clasa de toate celelalte
- Pentru predictie, alege clasa cu cel mai mare scor de incredere
- Timp de antrenare: $O(K \cdot n)$ unde $K$ este numarul de clase

**One-vs-One (OvO)**:
- Antreneaza $\frac{K(K-1)}{2}$ clasificatori binari (cate unul pentru fiecare pereche de clase)
- Fiecare clasificator separa doua clase specifice
- Pentru predictie, foloseste votarea: fiecare clasificator voteaza pentru o clasa
- Timp de antrenare: $O(K^2 \cdot n/K^2) = O(n)$, dar cu mai multe modele

`SVC` din scikit-learn foloseste implicit OvO pentru problemele multi-clasa.


In [ ]:
# Load digits dataset (10 classes: 0-9)
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Number of samples: {X_digits.shape[0]}")
print(f"Number of features: {X_digits.shape[1]}")
print(f"Number of classes: {len(np.unique(y_digits))}")

# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for idx, ax in enumerate(axes.flat):
    ax.imshow(digits.images[idx], cmap='gray')
    ax.set_title(f"Label: {digits.target[idx]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

# Split and scale
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
    X_digits, y_digits, test_size=0.25, random_state=42, stratify=y_digits
)

scaler_d = StandardScaler()
X_tr_d_s = scaler_d.fit_transform(X_tr_d)
X_te_d_s = scaler_d.transform(X_te_d)

<a name='ex-5'></a>
#### **Exercitiul 5 - `train_multiclass_svm_strategies`**

**Sarcina ta:**

Implementeaza functia `train_multiclass_svm_strategies` care antreneaza atat strategia One-vs-Rest, cat si One-vs-One si compara performanta lor.

Trebuie sa implementezi:

* **Antreneaza One-vs-Rest (OvR)**:
    * Foloseste wrapper-ul `OneVsRestClassifier` peste un `SVC`.
    * Antreneaza pe datele de train si prezice pe datele de test.
    * Calculeaza acuratetea pe test.

* **Antreneaza One-vs-One (OvO)**:
    * Foloseste wrapper-ul `OneVsOneClassifier` peste un `SVC`.
    * Antreneaza pe datele de train si prezice pe datele de test.
    * Calculeaza acuratetea pe test.

* **Returneaza rezultatele**:
    * Returneaza un dictionar cu ambele acurateti si ambele modele antrenate.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru One-vs-Rest:**
* Creeaza estimatorul de baza: `svc = SVC(kernel=kernel, C=C, gamma=gamma)`.
* Inveleste-l: `ovr_model = OneVsRestClassifier(svc)`.
* Antreneaza: `ovr_model.fit(X_train, y_train)`.
* Evalueaza: `ovr_acc = accuracy_score(y_test, ovr_model.predict(X_test))`.

**Pentru One-vs-One:**
* Similar cu OvR, dar foloseste: `ovo_model = OneVsOneClassifier(svc)`.

**Pentru dictionarul returnat:**
* `results = {'ovr_acc': ..., 'ovo_acc': ..., 'ovr_model': ..., 'ovo_model': ...}`.

</details>


In [ ]:
# GRADED FUNCTION: train_multiclass_svm_strategies
def train_multiclass_svm_strategies(X_train, y_train, X_test, y_test, C=2.0, kernel='rbf', gamma='scale'):
    """
    Train OvR and OvO SVM strategies for multi-class classification.

    Args:
        X_train: Training features
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        C: Regularization parameter
        kernel: Kernel type
        gamma: Kernel coefficient

    Returns:
        results: dict with keys:
            'ovr_acc': float - One-vs-Rest test accuracy
            'ovo_acc': float - One-vs-One test accuracy
            'ovr_model': trained OneVsRestClassifier
            'ovo_model': trained OneVsOneClassifier
    """
    results = {}
    
    ### ÎNCEPUT DE COD AICI ###
    
    # Create base SVC
    
    # Train One-vs-Rest
    
    # Train One-vs-One
    
    ### SFÂRȘIT DE COD AICI ###
    
    return results

In [ ]:
# Test your implementation
multi_results = train_multiclass_svm_strategies(X_tr_d_s, y_tr_d, X_te_d_s, y_te_d, C=2.0, kernel='rbf', gamma='scale')

print("="*60)
print("MULTI-CLASS SVM STRATEGIES")
print("="*60)
print(f"One-vs-Rest (OvR) accuracy: {multi_results['ovr_acc']:.4f}")
print(f"One-vs-One (OvO) accuracy:  {multi_results['ovo_acc']:.4f}")
print("="*60)

# Number of classifiers trained
n_classes = len(np.unique(y_tr_d))
n_ovr = n_classes
n_ovo = n_classes * (n_classes - 1) // 2

print(f"\nNumber of binary classifiers:")
print(f"  OvR: {n_ovr} classifiers (one per class)")
print(f"  OvO: {n_ovo} classifiers (one per class pair)")

##### **Rezultatul asteptat**
```
============================================================
MULTI-CLASS SVM STRATEGIES
============================================================
One-vs-Rest (OvR) accuracy: 0.9XXX
One-vs-One (OvO) accuracy:  0.9XXX
============================================================

Number of binary classifiers:
  OvR: 10 classifiers (one per class)
  OvO: 45 classifiers (one per class pair)
```


In [ ]:
unittests.exercise_5(train_multiclass_svm_strategies)

<a name='7'></a>
## 7 - SVM liniar de la zero

Acum ca intelegi cum functioneaza SVM-urile, hai sa implementam un SVM liniar de la zero folosind gradient descent. Acest lucru iti va aprofunda intelegerea procesului de optimizare din spatele SVM-urilor.


<a name='7-1'></a>
### 7.1 - Formularea primala

Vom rezolva problema primala pentru soft-margin SVM:

$$\min_{w,b} \frac{1}{2}\|w\|^2 + C\sum_{i=1}^{m}\max(0, 1 - y_i(w^T x_i + b))$$

Al doilea termen este **hinge loss**: $\ell(y, s) = \max(0, 1 - ys)$ unde $s = w^T x + b$ este scorul de decizie.

Vom folosi **subgradient descent** deoarece hinge loss nu este diferentiabila in punctul $ys = 1$. Subgradientul este:

Pentru termenul de regularizare: $\nabla_w \frac{1}{2}\|w\|^2 = w$

Pentru termenul hinge loss:
$$\nabla_w \ell = \begin{cases} -y_i x_i & \text{if } y_i(w^T x_i + b) < 1 \\ 0 & \text{otherwise} \end{cases}$$

$$\nabla_b \ell = \begin{cases} -y_i & \text{if } y_i(w^T x_i + b) < 1 \\ 0 & \text{otherwise} \end{cases}$$

**Nota**: Vom lucra cu etichete din $\{-1, +1\}$ pentru comoditate matematica.


<a name='ex-6a'></a>
#### **Exercitiul 6a - `hinge_loss`**

**Sarcina ta:**

Implementeaza functia `hinge_loss` care calculeaza hinge loss pentru SVM.

Trebuie sa implementezi:

* **Calculeaza hinge loss pentru fiecare exemplu**:
    * Pentru fiecare exemplu, calculeaza: $\max(0, 1 - y_i \cdot s_i)$
    * unde $y_i \in \{-1, +1\}$ iar $s_i$ este scorul de decizie.

* **Returneaza pierderea medie**:
    * Fa media pierderilor pe toate exemplele.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru hinge loss:**
* Calculeaza incalcarea marginii: `margin = 1 - y_true * scores`.
* Aplica max: `loss = np.maximum(0, margin)`.
* Returneaza media: `return np.mean(loss)`.

**Alternativa vectorizata:**
* Intr-o singura linie: `return np.mean(np.maximum(0, 1 - y_true * scores))`.

</details>


In [ ]:
# GRADED FUNCTION: hinge_loss
def hinge_loss(y_true, scores):
    """
    Compute hinge loss: mean(max(0, 1 - y*s))

    Args:
        y_true: numpy array of shape (m,) - labels in {-1, +1}
        scores: numpy array of shape (m,) - raw decision scores (w^T x + b)

    Returns:
        loss: float - mean hinge loss
    """
    ### ÎNCEPUT DE COD AICI ###
    
    # Calculate hinge loss for each sample
    
    # Return mean loss
    
    ### SFÂRȘIT DE COD AICI ###
    
    return loss

In [ ]:
# Test hinge_loss
y_test = np.array([-1, 1, 1, -1, 1])
scores_test = np.array([-2.0, 1.5, 0.3, 1.0, -0.5])

loss = hinge_loss(y_test, scores_test)
print(f"Hinge loss: {loss:.4f}")

# Test perfect predictions
scores_perfect = np.array([-2.0, 2.0, 2.0, -2.0, 2.0])
loss_perfect = hinge_loss(y_test, scores_perfect)
print(f"Hinge loss (perfect predictions): {loss_perfect:.4f}")

##### **Rezultatul asteptat**
```
Hinge loss: 0.6600
Hinge loss (perfect predictions): 0.0000
```


In [ ]:
unittests.exercise_6a(hinge_loss)

<a name='ex-6b'></a>
#### **Exercitiul 6b - `svm_primal_gradients`**

**Sarcina ta:**

Implementeaza functia `svm_primal_gradients` care calculeaza subgradientii functiei obiectiv primale a SVM-ului.

Trebuie sa implementezi:

* **Calculeaza scorurile de decizie**:
    * Calculeaza $s_i = w^T x_i + b$ pentru toate exemplele.

* **Identifica incalcarile de margine**:
    * Gaseste exemplele pentru care $y_i \cdot s_i < 1$ (acestea contribuie la gradient).

* **Calculeaza gradientul pentru w**:
    * Porneste de la termenul de regularizare: `grad_w = w`.
    * Adauga gradientul hinge loss: pentru exemplele care incalca marginea, scade $C \cdot y_i \cdot x_i$.

* **Calculeaza gradientul pentru b**:
    * Aduna $-C \cdot y_i$ pentru toate exemplele care incalca marginea.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru scoruri si incalcari:**
* Scoruri: `scores = X @ w + b`.
* Masca de incalcare: `violated = (y * scores) < 1`.

**Pentru `grad_w`:**
* Porneste cu regularizarea: `grad_w = w.copy()`.
* Adauga contributia hinge: `grad_w -= C * (X[violated].T @ y[violated]) / len(y)`.
* Sau versiunea cu bucla:
  ```python
  for i in range(len(y)):
      if violated[i]:
          grad_w -= C * y[i] * X[i] / len(y)
  ```

**Pentru `grad_b`:**
* `grad_b = -C * np.sum(y[violated]) / len(y)`.

</details>


In [ ]:
# GRADED FUNCTION: svm_primal_gradients
def svm_primal_gradients(X, y, w, b, C):
    """
    Compute subgradients for the SVM primal objective:
        0.5*||w||^2 + C * sum_i max(0, 1 - y_i (w^T x_i + b))

    Args:
        X: numpy array of shape (m, n) - features
        y: numpy array of shape (m,) - labels in {-1, +1}
        w: numpy array of shape (n,) - weight vector
        b: float - bias term
        C: float - regularization parameter

    Returns:
        grad_w: numpy array of shape (n,) - gradient with respect to w
        grad_b: float - gradient with respect to b
    """
    ### ÎNCEPUT DE COD AICI ###
    
    # Compute decision scores
    
    # Identify margin violations
    
    # Compute gradient for w
    
    # Compute gradient for b
    
    ### SFÂRȘIT DE COD AICI ###
    
    return grad_w, grad_b

In [ ]:
# Test svm_primal_gradients
X_test = np.array([[1, 2], [2, 3], [3, 1], [2, 1]])
y_test = np.array([-1, -1, 1, 1])
w_test = np.array([0.5, 0.5])
b_test = 0.0
C_test = 1.0

grad_w, grad_b = svm_primal_gradients(X_test, y_test, w_test, b_test, C_test)
print(f"Gradient w.r.t. w: {grad_w}")
print(f"Gradient w.r.t. b: {grad_b:.4f}")

##### **Rezultatul asteptat** (valorile pot varia usor)
```
Gradient w.r.t. w: [X.XXXX X.XXXX]
Gradient w.r.t. b: X.XXXX
```


In [ ]:
unittests.exercise_6b(svm_primal_gradients)

<a name='ex-6c'></a>
#### **Exercitiul 6c - `train_linear_svm_from_scratch`**

**Sarcina ta:**

Implementeaza functia `train_linear_svm_from_scratch` care antreneaza un SVM liniar folosind gradient descent.

Trebuie sa implementezi:

* **Converteaza etichetele la {-1, +1}**:
    * Daca etichetele sunt in {0, 1}, converteste-le: $y_{new} = 2 \cdot y_{old} - 1$.

* **Initializeaza parametrii**:
    * Initializeaza `w` cu zerouri: `np.zeros(n_features)`.
    * Initializeaza `b` cu zero.

* **Bucla de gradient descent**:
    * Pentru fiecare iteratie:
        * Calculeaza gradientii folosind `svm_primal_gradients`.
        * Actualizeaza parametrii: $w \leftarrow w - \alpha \cdot \nabla_w$ si $b \leftarrow b - \alpha \cdot \nabla_b$.
        * Calculeaza si stocheaza valoarea functiei obiectiv: $\frac{1}{2}\|w\|^2 + C \cdot \text{hinge\_loss}$.

* **Returneaza rezultatele**:
    * Returneaza `w`, `b` si istoricul valorilor functiei obiectiv.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru conversia etichetelor:**
* Verifica daca sunt binare: `if np.all(np.isin(y, [0, 1]))`.
* Converteste: `y_train = 2 * y - 1` (transforma 0→-1, 1→+1).

**Pentru initializare:**
* `w = np.zeros(X.shape[1])`.
* `b = 0.0`.

**Pentru bucla de antrenare:**
```python
for iteration in range(num_iters):
    grad_w, grad_b = svm_primal_gradients(X, y_train, w, b, C)
    w -= lr * grad_w
    b -= lr * grad_b

    scores = X @ w + b
    reg_term = 0.5 * np.dot(w, w)
    hinge_term = C * hinge_loss(y_train, scores)
    objective = reg_term + hinge_term
    history.append(objective)
```

</details>


In [ ]:
# GRADED FUNCTION: train_linear_svm_from_scratch
def train_linear_svm_from_scratch(X, y, C=1.0, lr=0.01, num_iters=2000, verbose=True):
    """
    Train a linear SVM using subgradient descent.

    Args:
        X: numpy array of shape (m, n) - training features
        y: numpy array of shape (m,) - labels in {0,1} or {-1,+1}
        C: float - regularization parameter
        lr: float - learning rate
        num_iters: int - number of gradient descent iterations
        verbose: bool - whether to print progress

    Returns:
        w: numpy array of shape (n,) - learned weight vector
        b: float - learned bias term
        history: list - objective value at each iteration
    """
    history = []
    
    ### ÎNCEPUT DE COD AICI ###
    
    # Convert labels to {-1, +1} if necessary
    
    # Initialize parameters
    
    # Gradient descent loop
    
    ### SFÂRȘIT DE COD AICI ###
    
    if verbose and (num_iters >= 100):
        print(f"Iteration {num_iters}: Objective = {history[-1]:.4f}")
    
    return w, b, history

In [ ]:
# Test on linear dataset with scaling
X_tr_scratch, X_te_scratch, y_tr_scratch, y_te_scratch = train_test_split(
    X_lin, y_lin, test_size=0.25, random_state=42, stratify=y_lin
)

# Scale features (important!)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr_scratch)
X_te_s = scaler.transform(X_te_scratch)

# Train from scratch
w, b, hist = train_linear_svm_from_scratch(X_tr_s, y_tr_scratch, C=1.0, lr=0.01, num_iters=2000, verbose=True)

# Helper function for prediction
def predict_linear_svm_from_scratch(X, w, b):
    """Predict using learned SVM parameters."""
    scores = X @ w + b
    predictions = (scores >= 0).astype(int)
    return predictions, scores

# Evaluate
y_pred_s, scores_s = predict_linear_svm_from_scratch(X_te_s, w, b)
accuracy = accuracy_score(y_te_scratch, y_pred_s)

print(f"\nFrom-scratch SVM accuracy: {accuracy:.4f}")

# Compare with scikit-learn
sklearn_model = train_linear_svm(X_tr_s, y_tr_scratch, C=1.0)
sklearn_acc = accuracy_score(y_te_scratch, sklearn_model.predict(X_te_s))
print(f"scikit-learn SVM accuracy: {sklearn_acc:.4f}")

##### **Rezultatul asteptat**
```
Iteration 2000: Objective = X.XXXX

From-scratch SVM accuracy: 0.9XXX
scikit-learn SVM accuracy: 1.0000
```


In [ ]:
unittests.exercise_6c(train_linear_svm_from_scratch)

In [ ]:
# Visualize training progress
plt.figure(figsize=(10, 5))
plt.plot(hist)
plt.title("Training Objective Over Time")
plt.xlabel("Iteration")
plt.ylabel("Objective Value")
plt.grid(True, alpha=0.3)
plt.show()

# Visualize decision boundary
class ScratchModel:
    """Wrapper for plotting compatibility."""
    def __init__(self, w, b):
        self.w = w
        self.b = b
    def predict(self, X):
        return (X @ self.w + self.b >= 0).astype(int)

plot_decision_boundary_2d(ScratchModel(w, b), X_tr_s, y_tr_scratch, 
                         title="From-Scratch Linear SVM (Scaled Data)")

## Felicitari!

Ai finalizat cu succes tema despre Support Vector Machine! Ai construit o intelegere cuprinzatoare a unuia dintre cei mai puternici si mai bine fundamentati teoretic algoritmi de machine learning.

### Ce ai realizat:

1. **Ai inteles teoria SVM**: Ai invatat despre maximizarea marginii, vectorii-suport si trucul kernelului
2. **Ai implementat un SVM liniar**: Ai antrenat clasificatori pe date liniar separabile si ai vizualizat frontierele de decizie
3. **Ai demonstrat importanta scalarii trasaturilor**: Ai aratat cat de critica este normalizarea trasaturilor pentru performanta SVM
4. **Ai aplicat kernelul RBF**: Ai folosit kerneluri neliniare pentru a trata frontiere de decizie complexe
5. **Ai comparat kerneluri**: Ai evaluat kerneluri liniare, polinomiale si RBF pe mai multe seturi de date
6. **Ai implementat strategii multi-clasa**: Ai implementat si comparat abordarile One-vs-Rest si One-vs-One
7. **Ai construit SVM de la zero**: Ai implementat un SVM liniar complet folosind gradient descent
8. **Ai ajustat hiperparametri**: Ai folosit grid search pentru a gasi valorile optime ale lui C si gamma

### Idei-cheie:

* **SVM-urile maximizeaza marginea**: Aceasta abordare geometrica conduce la frontiere de decizie robuste
* **Vectorii-suport sunt critici**: Doar un subset mic din punctele de antrenare defineste modelul
* **Scalarea trasaturilor este esentiala**: SVM-urile sunt bazate pe distante si sunt foarte sensibile la scale diferite
* **Kernelurile permit neliniaritate**: Trucul kernelului ofera flexibilitate fara transformare explicita de trasaturi
* **Ajustarea hiperparametrilor conteaza**: C si gamma influenteaza semnificativ performanta modelului
* **Strategiile multi-clasa**: OvR si OvO extind clasificatorii binari la probleme multi-clasa

### Cand sa folosesti SVM-uri:

**Potrivite pentru:**
- Seturi de date de dimensiune medie (sute pana la mii de exemple)
- Spatii cu dimensionalitate mare (text, genomica)
- Situatii in care exista o margine de separare clara
- Nevoia de frontiere de decizie interpretabile
- Cazuri in care datele de antrenare sunt limitate

**Ia in considerare alternative pentru:**
- Seturi de date foarte mari (>100K exemple) - antrenarea poate fi lenta
- Date foarte zgomotoase, cu clase suprapuse
- Situatii in care ai nevoie de estimari de probabilitate (necesita calibrare)
- Domenii in care deep learning exceleaza (imagini, vorbire, NLP cu volume masive de date)

**Foarte bine! Acum ai o baza solida in Support Vector Machines!**
